# 08 – Validation sheet, agreement script and the dataset's own labels

1. Build the 50-chunk validation sheet (25 chatbot answers, 25 news chunks) so that a human can label which frame each text emphasises most, **without seeing the dictionary's answer**, and check that the files open cleanly.
2. The agreement script (`agreement.py`): dictionary against human labels, with per-frame precision, recall and Cohen's kappa. Tested here on dummy labels.
3. Compare the dictionary's news scores with the **dataset's own frame labels** (`brewcoua/genai-gdelt-framing`), and list disagreements.

**Labelling is left to the team.** Nothing in this notebook produces human labels; the dummy labels in section 3 exist only to test the script and are never saved.

In [1]:
import bisect, json, re, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from framescore import Lexicon, find_hits, count_vector
from agreement import agreement_report, cohen_kappa, per_class_report, confusion
from sklearn.metrics import cohen_kappa_score

CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); FR = Path("dataset/frames"); DATA = Path("data")
SEED = 42
FRAMES = list(json.load(open(CFG["lexicon_json"]))["frames"]); HC = [f"hits_{f}" for f in FRAMES]
LABELS = FRAMES + ["none"]
LEX = Lexicon(CFG["lexicon_json"])
# the corpus on-topic rule (the same two regexes as in notebooks 04, 05 and the corpus builder): at least 2 AI words and 2 regulation words
AI_WORDS  = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b")
REG_WORDS = re.compile(r"\b(regulat\w*|legislat\w*|laws?|AI Act|bills?|rules?|oversight|ban|governance|policy|lawmakers?)\b", re.I)

## 1. The validation sheet
- **Stratified:** chatbot answers 13 Llama + 12 Qwen; news chunks 7 + 6 + 6 + 6 across the four regions. Random within each stratum (seeded).
- **No leaks.** The sheet the labeller gets has only an opaque `label_id`, the text and an empty `human_frame` column: no prompt id (it carries the wording), no source, no lexicon label, no hit counts. A separate **key** file maps `label_id` back to the source and holds the dictionary's top frame. The dictionary's top frame is the frame with the most hits; ties go to the frame listed first in the lexicon; `none` when the text has no hits.
- **Instruction row:** `label_id = 0` is an instruction row (ignored by the scoring script), so the sheet is self-explaining when opened in a spreadsheet. Full definitions are in `dataset/frames/validation_instructions.md`.
- **How the 50 texts were sampled (audited 2026-10-29, news excerpts redrawn the same day).** Seeded random sampling (`random_state = 42`) **within strata**, from all 216 answers and all 420 news articles: 13 Llama and 12 Qwen answers; 7 US, 6 UK, 6 EU and 6 Other articles. The draw of answers and articles uses only the row positions and the stratum label: it did **not** use lexicon hits, hit counts or frame scores, so it does not depend on the lexicon version (checked below: zeroing every hit count gives the same draw).
- **What the labeller sees.** A chatbot text is the **first 300 words** of the answer (unchanged). A news text is a **window of 250 to 300 words from inside the article**, not its opening. **Window rule:** a target length is drawn at random between 250 and 300 words (seeded); among all windows of that length the one with the most AI-word plus regulation-word matches (`AI_WORDS` and `REG_WORDS`, the corpus on-topic regexes) is chosen, subject to the window itself passing the on-topic rule (at least 2 AI words and 2 regulation words inside it); ties go to the earliest window. **No frame hit and no lexicon term is used to choose the window.** If no window of the drawn length passes the rule, the window with the most matches is used and flagged (`passes_rule = False` in `dataset/analysis/validation_news_windows.csv`). The key's dictionary label and hit count are computed on exactly the text shown.
- **Why windows.** The main H1 analysis scores texts in full (decision 2026-10-29), and the opening of an article is not representative of its whole (only 37% of corpus articles still satisfy the on-topic rule within their first 300 words). A topic-selected window is a fairer excerpt for a human to label, but it is still an excerpt of the article, not the full text the dictionary scores in the main analysis.
- **Safe to rerun.** `REBUILD_SHEET = False` keeps the existing sheet files (so labels already collected still match their `label_id`s); the audit in section 1b compares them with a fresh draw and with the current corpus.

In [2]:
def head_words(text, n=300):
    for k, m in enumerate(re.finditer(r"\S+", text)):
        if k == n - 1:
            return text[:m.end()]
    return text

ans = pd.read_json(CFG["answers_jsonl"], lines=True); ans = ans[ans["chatbot"].isin(["llama3.1:8b", "qwen3.5:9b"])].reset_index(drop=True)
sc_a = pd.read_csv(OUT / "frame_scores_chatbot.csv"); assert (sc_a["prompt_id"] == ans["prompt_id"]).all()
news = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False); sc_n = pd.read_csv(OUT / "frame_scores_news.csv"); assert (sc_n["doc_id"] == news["doc_id"]).all()

def top_frame(row):
    h = row[HC].to_numpy()
    return "none" if h.sum() == 0 else FRAMES[int(np.argmax(h))]          # argmax: ties go to the first frame in the lexicon order

def pick(df, by, sizes):
    return pd.concat([df[df[by] == k].sample(n, random_state=SEED) for k, n in sizes.items()])

a_pick = pick(sc_a.assign(i=range(len(sc_a))), "chatbot", {"llama3.1:8b": 13, "qwen3.5:9b": 12})
n_pick = pick(sc_n.assign(i=range(len(sc_n))), "region", {"US": 7, "UK": 6, "EU": 6, "Other": 6})

def topic_window(text, L):
    """The window of L words (the whole text if it is shorter) with the most AI-word + regulation-word matches among the windows that
    pass the on-topic rule (at least 2 of each inside the window); ties go to the earliest window. If no window passes, the window with
    the most matches is returned and flagged. No frame hit or lexicon term is used."""
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]; n = len(spans); L = min(L, n); starts = [a for a, _ in spans]
    def cum(rx):
        c = np.zeros(n + 1)
        for m in rx.finditer(text): c[bisect.bisect_right(starts, m.start())] += 1
        return np.cumsum(c)
    ca, cr = cum(AI_WORDS), cum(REG_WORDS)
    s = np.arange(0, n - L + 1); ai, rg = ca[s + L] - ca[s], cr[s + L] - cr[s]
    ok = (ai >= 2) & (rg >= 2); score = np.where(ok, ai + rg, -1) if ok.any() else ai + rg
    j = int(score.argmax())
    return {"start_word": j, "words": L, "ai_words": int(ai[j]), "reg_words": int(rg[j]), "passes_rule": bool(ok[j]), "text": text[spans[j][0]: spans[j + L - 1][1]]}

target = np.random.default_rng(SEED).integers(250, 301, size=len(n_pick))
rows = [{"source": "chatbot", "ref": f"{ans.loc[r.i, 'prompt_id']}|r{ans.loc[r.i, 'replicate']}|{ans.loc[r.i, 'chatbot']}", "text": head_words(ans.loc[r.i, "answer"])} for _, r in a_pick.iterrows()]
win_rows = []
for (_, r), L in zip(n_pick.iterrows(), target):
    full = news.loc[r.i, "text"]; w = topic_window(full, int(L))
    rows.append({"source": "news", "ref": news.loc[r.i, "doc_id"], "text": w["text"]})
    win_rows.append({"doc_id": news.loc[r.i, "doc_id"], "region": news.loc[r.i, "region"], "article_words": len(full.split()), "target_words": int(L),
                     **{k: w[k] for k in ("start_word", "words", "ai_words", "reg_words", "passes_rule")}})
win_info = pd.DataFrame(win_rows).sort_values("doc_id").reset_index(drop=True)
win_info.to_csv(OUT / "validation_news_windows.csv", index=False)         # no text, no label_id: safe to commit and keeps the sheet blind

sheet = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet.insert(0, "label_id", range(1, len(sheet) + 1))
# dictionary label and hit count of the text exactly as shown (whitespace flattened, as the scorer does), lexicon v1.2
V = np.array([count_vector(h, FRAMES) for h in find_hits([" ".join(t.split()) for t in sheet["text"]], LEX.matcher, LEX.owner)])
sheet["lexicon_hits"] = V.sum(axis=1)
sheet["lexicon_top_frame"] = ["none" if v.sum() == 0 else FRAMES[int(np.argmax(v))] for v in V]     # argmax: ties go to the first frame in the lexicon order

INSTR = ("INSTRUCTIONS (row 0, ignore when scoring): read each text and put in human_frame the ONE frame it emphasises most: "
         + ", ".join(FRAMES) + ", or none (no clear frame). Judge emphasis, not keywords. Do not discuss labels with the other labellers. Definitions: dataset/frames/validation_instructions.md")
blank = pd.concat([pd.DataFrame([{"label_id": 0, "text": INSTR, "human_frame": ""}]), sheet[["label_id", "text"]].assign(human_frame="")], ignore_index=True)
key = sheet[["label_id", "source", "ref", "lexicon_top_frame", "lexicon_hits"]]
fresh_blank, fresh_key = blank, key                                         # the draw as it comes out of the current files
REBUILD_SHEET = False                                                       # True overwrites data/validation_sheet_*.csv; leave False once anyone has started labelling
if (DATA / "validation_sheet_blank.csv").exists() and (DATA / "validation_sheet_key.csv").exists() and not REBUILD_SHEET:
    blank = pd.read_csv(DATA / "validation_sheet_blank.csv", keep_default_na=False); key = pd.read_csv(DATA / "validation_sheet_key.csv", keep_default_na=False)
    print("kept the existing sheet and key (REBUILD_SHEET is False); a fresh draw is compared with them in section 1b")
else:
    blank.to_csv(DATA / "validation_sheet_blank.csv", index=False); key.to_csv(DATA / "validation_sheet_key.csv", index=False); print("sheet and key written")
print(len(key), "chunks:", key["source"].value_counts().to_dict(), "| dictionary top frame in the key:", key["lexicon_top_frame"].value_counts().to_dict())

kept the existing sheet and key (REBUILD_SHEET is False); a fresh draw is compared with them in section 1b
50 chunks: {'chatbot': 25, 'news': 25} | dictionary top frame in the key: {'regulation_governance': 14, 'economy_labour': 13, 'risk_safety': 10, 'innovation_opportunity': 8, 'rights_privacy': 3, 'misinformation_integrity': 1, 'none': 1}


In [3]:
defs = '''# Validation sheet: how to label

For each text choose the ONE frame it emphasises most (what most of the text is about), or `none`.
Read the text, do not search for keywords. Use only the text in the sheet. Fill in the `human_frame` column with the exact frame name.

| frame | choose it when the text mainly talks about |
|---|---|
| innovation_opportunity | benefits, breakthroughs, progress, new products, opportunity, startups |
| risk_safety | harms, dangers, accidents, misuse, safety testing, threats, children harmed |
| regulation_governance | regulators, oversight, enforcement, compliance, treaties, standards, legal responsibility, institutions |
| rights_privacy | privacy, personal data, surveillance, discrimination, civil liberties, copyright |
| economy_labour | jobs, workers, automation, markets, competition between firms or countries, costs, trade |
| misinformation_integrity | false information, deepfakes, hallucinations, manipulation, election integrity, authenticity |
| none | none of these clearly dominates (for example a product review or a personal essay) |

Rules: one label per text; if two frames seem equal choose the one that appears first in the text; do not look at the key file until everyone has finished.

**Human review vs human oversight.** A person checking a machine: human review of automated decisions counts as rights_privacy; human oversight or human intervention as a control counts as regulation_governance. (Example: "people should have the right to a human review of an AI decision" is rights_privacy; "a human must be able to intervene and switch the system off" is regulation_governance.)

**How the 50 texts were chosen.** 25 chatbot answers (13 Llama, 12 Qwen) and 25 news articles (7 US, 6 UK, 6 EU, 6 Other), drawn at random within those groups (seeded); the choice did not use the dictionary or any frame score. A chatbot text is the first 300 words of the answer. A news text is a **window of 250 to 300 words from inside the article**, not its opening. **Window rule:** a target length is drawn at random between 250 and 300 words (seeded); among all windows of that length, the one with the most AI words (AI, artificial intelligence, chatbot, ChatGPT, OpenAI, LLM, ...) and regulation words (regulat-, legislat-, law, rule, bill, ban, oversight, governance, policy, lawmaker) is chosen, provided it has at least two of each; ties go to the earliest window. If no window of that length has two of each, the window with the most such words is used and flagged in the analysis files (two articles are in this case). No frame hit or dictionary term is used to choose a window. A window is a continuous passage, so it may start or end in the middle of an argument: label what you see, not what the article might say before or after.

**Content note.** Some texts discuss suicide, sexual content in the context of chatbot safety, and hate speech. You may skip any text that distresses you: write `skip` in the label column (the scoring scripts treat it as missing and report how many were skipped) and tell the others.
'''
(FR / "validation_instructions.md").write_text(defs)
print("written: dataset/frames/validation_instructions.md")

written: dataset/frames/validation_instructions.md


### Does the sheet open cleanly?

In [4]:
b = pd.read_csv(DATA / "validation_sheet_blank.csv", keep_default_na=False); k = pd.read_csv(DATA / "validation_sheet_key.csv", keep_default_na=False)
checks = {
 "blank sheet has exactly the columns label_id, text, human_frame": list(b.columns) == ["label_id", "text", "human_frame"],
 "instruction row present (label_id 0) and the label column is empty everywhere": b.iloc[0]["label_id"] == 0 and b["human_frame"].eq("").all(),
 "50 data rows + 1 instruction row": len(b) == 51 and b["label_id"].iloc[1:].tolist() == list(range(1, 51)),
 "label ids unique": b["label_id"].is_unique,
 "no empty texts; texts are at most 300 words": b["text"].str.len().gt(0).all() and b["text"].iloc[1:].map(lambda t: len(t.split())).le(300).all(),
 "blank sheet leaks nothing: no source, prompt id, model name or lexicon column": not set(b.columns) & {"source", "ref", "lexicon_top_frame", "lexicon_hits", "prompt_id", "chatbot"},
 "no prompt id, model name or doc id inside the texts": not b["text"].iloc[1:].str.contains(r"Q\d\d_|llama3|qwen3|news_\d{4}", regex=True).any(),
 "key is a separate file with the mapping": list(k.columns) == ["label_id", "source", "ref", "lexicon_top_frame", "lexicon_hits"] and k["label_id"].tolist() == list(range(1, 51)),
 "key frame labels are valid": k["lexicon_top_frame"].isin(LABELS).all(),
}
for name, ok in checks.items(): print("OK  " if ok else "FAIL", name)
assert all(checks.values())
b.drop(columns="text").head(4)      # text column not shown: article text is not kept in committed notebook outputs

OK   blank sheet has exactly the columns label_id, text, human_frame
OK   instruction row present (label_id 0) and the label column is empty everywhere
OK   50 data rows + 1 instruction row
OK   label ids unique
OK   no empty texts; texts are at most 300 words
OK   blank sheet leaks nothing: no source, prompt id, model name or lexicon column
OK   no prompt id, model name or doc id inside the texts
OK   key is a separate file with the mapping
OK   key frame labels are valid


,label_id,human_frame
0,0,
1,1,
2,2,
3,3,


### 1b. Audit against the current corpus and lexicon (2026-10-29)
Questions, answered from the files (nothing rewritten): (1) is the sheet the draw this code produces; (2) is each chatbot text the first 300 words of the current answer, and each news text a verbatim 250 to 300 word passage of the current article; (3) does every news window pass the on-topic rule, and how many words does each have; (4) are the dictionary labels in the key what the current lexicon (v1.2) gives on the text shown; (5) does the draw depend on hit counts?

In [5]:
k = key.copy()
same_refs = set(fresh_key["ref"]) == set(k["ref"]); same_ids = fresh_key["label_id"].tolist() == k["label_id"].tolist()
texts_now = {r.label_id: r.text for r in fresh_blank.iloc[1:].itertuples()}
text_match = int(sum(blank.set_index("label_id").loc[l, "text"] == t for l, t in texts_now.items()))
# (2) each shown text against the current corpus
shown = blank.set_index("label_id")["text"]
chat_ok = all(shown[r.label_id] == head_words(ans[(ans["prompt_id"] == r.ref.split("|")[0]) & (ans["replicate"] == int(r.ref.split("|")[1][1:])) & (ans["chatbot"] == r.ref.split("|")[2])].iloc[0]["answer"])
              for r in k[k["source"] == "chatbot"].itertuples())
art = news.set_index("doc_id")["text"]
news_ok = all(shown[r.label_id] in art[r.ref] for r in k[k["source"] == "news"].itertuples())
nw = k[k["source"] == "news"].assign(words=lambda d: [len(shown[l].split()) for l in d["label_id"]])
on_topic = [len(AI_WORDS.findall(shown[l])) >= 2 and len(REG_WORDS.findall(shown[l])) >= 2 for l in nw["label_id"]]
# (5) does the draw use the hit counts? re-draw with every hit count set to zero
zero_a, zero_n = sc_a.assign(i=range(len(sc_a)), hits_total=0), sc_n.assign(i=range(len(sc_n)), hits_total=0)
zeroed = (pick(zero_a, "chatbot", {"llama3.1:8b": 13, "qwen3.5:9b": 12})["i"].tolist() == a_pick["i"].tolist()
          and pick(zero_n, "region", {"US": 7, "UK": 6, "EU": 6, "Other": 6})["i"].tolist() == n_pick["i"].tolist())
cur = fresh_key.set_index("label_id")
diff_top = int((k.set_index("label_id")["lexicon_top_frame"] != cur["lexicon_top_frame"]).sum())
diff_hits = int((k.set_index("label_id")["lexicon_hits"].astype(int) != cur["lexicon_hits"].astype(int)).sum())
print("sheet refs identical to a fresh draw:", same_refs, "| same label ids:", same_ids, "| texts identical to a fresh draw:", text_match, "of", len(texts_now))
print("chatbot texts = first 300 words of the current answers:", chat_ok, "| news texts are verbatim passages of the current articles:", news_ok)
print(f"news windows: {len(nw)} | words min {nw['words'].min()} / median {int(nw['words'].median())} / max {nw['words'].max()} | pass the on-topic rule: {sum(on_topic)} of {len(nw)}")
print("news windows that do NOT pass the rule (no 250-300 word window of the article holds 2 AI and 2 regulation words):", nw.loc[[not o for o in on_topic], "ref"].tolist() or "none")
print("window words per news text:", sorted(nw["words"].tolist()))
print("draw unchanged when every hit count is set to zero (the sampling ignores lexicon hits):", zeroed)
print(f"key vs the current lexicon v1.2 on the text shown: top frame differs in {diff_top} of {len(k)}, hit count differs in {diff_hits} of {len(k)}")
assert same_refs and same_ids and text_match == len(texts_now) and chat_ok and news_ok and zeroed and diff_top == 0 and diff_hits == 0
assert nw["words"].between(250, 300).all(), "a news window is outside 250-300 words"

sheet refs identical to a fresh draw: True | same label ids: True | texts identical to a fresh draw: 50 of 50
chatbot texts = first 300 words of the current answers: True | news texts are verbatim passages of the current articles: True
news windows: 25 | words min 254 / median 276 / max 299 | pass the on-topic rule: 23 of 25
news windows that do NOT pass the rule (no 250-300 word window of the article holds 2 AI and 2 regulation words): ['news_0307', 'news_0282']
window words per news text: [254, 254, 254, 256, 259, 260, 268, 272, 272, 272, 275, 276, 276, 283, 285, 286, 287, 288, 289, 289, 290, 292, 293, 297, 299]
draw unchanged when every hit count is set to zero (the sampling ignores lexicon hits): True
key vs the current lexicon v1.2 on the text shown: top frame differs in 0 of 50, hit count differs in 0 of 50


## 2. The agreement script, tested on dummy labels
`agreement.py` holds `agreement_report` (accuracy, Cohen's kappa, per-frame precision/recall/F1, confusion matrix). The checks use cases whose answers can be worked out by hand and compare kappa with scikit-learn's.

In [6]:
# 1) identical labels
y = np.array(["a", "b", "c", "a", "b", "c"]); r = agreement_report(y, y, ["a", "b", "c"])
assert r["accuracy"] == 1 and abs(r["kappa"] - 1) < 1e-12 and (r["per_class"][["precision", "recall"]] == 1).all().all()

# 2) worked example: 10 items, 8 agree, each label used 5 times by both -> p_o = 0.8, p_e = 0.5, kappa = 0.6; precision = recall = 0.8
h = np.array(["A"] * 5 + ["B"] * 5); d = np.array(["A"] * 4 + ["B"] * 5 + ["A"])
r = agreement_report(h, d, ["A", "B"]); assert abs(r["accuracy"] - 0.8) < 1e-12 and abs(r["kappa"] - 0.6) < 1e-12
assert np.allclose(r["per_class"][["precision", "recall"]].to_numpy(), 0.8)

# 3) worked example with three labels and unequal use: checked by hand against p_o and p_e
h = np.array(["x"] * 6 + ["y"] * 3 + ["z"] * 1); d = np.array(["x"] * 5 + ["y"] * 4 + ["z"] * 1)   # one x was called y
p_o = 9 / 10; p_e = (6 / 10) * (5 / 10) + (3 / 10) * (4 / 10) + (1 / 10) * (1 / 10)
assert abs(cohen_kappa(h, d) - (p_o - p_e) / (1 - p_e)) < 1e-12

# 4) agrees with scikit-learn on random labels; independent random labels give kappa near 0
rng = np.random.default_rng(SEED); a = rng.choice(LABELS, 5000); c = rng.choice(LABELS, 5000)
assert abs(cohen_kappa(a, c, LABELS) - cohen_kappa_score(a, c)) < 1e-12 and abs(cohen_kappa(a, c, LABELS)) < 0.05
# 5) a label the checked side never uses -> recall 0, precision undefined (NaN), no crash
r = agreement_report(["a", "b"], ["a", "a"], ["a", "b"]); assert r["per_class"].loc[1, "recall"] == 0 and np.isnan(r["per_class"].loc[1, "precision"])
print("agreement script: 5 tests passed (identical, two worked examples, scikit-learn parity and chance level, empty class)")

agreement script: 5 tests passed (identical, two worked examples, scikit-learn parity and chance level, empty class)


In [7]:
def score_sheet(human_csv, key_csv=DATA / "validation_sheet_key.csv", save=True, tag="validation"):
    """Dictionary vs human labels. Reads the labelled sheet and the key, ignores the instruction row, checks every label is valid."""
    h = pd.read_csv(human_csv, keep_default_na=False); h = h[h["label_id"] != 0]
    skipped = int((h["human_frame"] == "skip").sum()); h = h[h["human_frame"] != "skip"]          # 'skip' = the labeller skipped a distressing text
    bad = sorted(set(h["human_frame"]) - set(LABELS))
    if bad or (h["human_frame"] == "").any():
        raise ValueError(f"empty or invalid labels: {bad}")
    m = h.merge(pd.read_csv(key_csv, keep_default_na=False), on="label_id")
    rep = agreement_report(m["human_frame"], m["lexicon_top_frame"], LABELS); rep["skipped"] = skipped
    by_source = {s: agreement_report(g["human_frame"], g["lexicon_top_frame"], LABELS) for s, g in m.groupby("source")}
    if save:
        rep["per_class"].round(3).to_csv(OUT / f"{tag}_per_frame.csv", index=False); rep["confusion"].to_csv(OUT / f"{tag}_confusion.csv")
        pd.DataFrame([{"scope": "all", "n": rep["n"], "skipped": skipped, "accuracy": rep["accuracy"], "kappa": rep["kappa"]}] +
                     [{"scope": s, "n": r["n"], "accuracy": r["accuracy"], "kappa": r["kappa"]} for s, r in by_source.items()]).round(3).to_csv(OUT / f"{tag}_summary.csv", index=False)
    return rep, by_source

# end-to-end test with DUMMY human labels (a fake labeller who agrees with the dictionary 70% of the time); written to a temp file, never saved in the repo
rng = np.random.default_rng(SEED); kk = pd.read_csv(DATA / "validation_sheet_key.csv", keep_default_na=False)
fake = [t if rng.random() < 0.7 else rng.choice(LABELS) for t in kk["lexicon_top_frame"]]
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "dummy.csv"; pd.concat([blank.iloc[:1], blank.iloc[1:].assign(human_frame=fake)]).to_csv(p, index=False)
    rep, by_source = score_sheet(p, save=False)
with tempfile.TemporaryDirectory() as td:               # a skipped text is dropped and counted, not an error
    p2 = Path(td) / "dummy_skip.csv"; sk = fake[:]; sk[0] = "skip"; sk[7] = "skip"
    pd.concat([blank.iloc[:1], blank.iloc[1:].assign(human_frame=sk)]).to_csv(p2, index=False)
    r_skip, _ = score_sheet(p2, save=False); assert r_skip["n"] == 48 and r_skip["skipped"] == 2
print("dummy labels: accuracy", round(rep["accuracy"], 2), "| kappa", round(rep["kappa"], 2), "| n", rep["n"], "(not saved; the pipeline works end to end)")
rep["per_class"].round(2)

dummy labels: accuracy 0.7 | kappa 0.63 | n 50 (not saved; the pipeline works end to end)


,label,n_reference,n_checked,true_positive,precision,recall,f1
0,innovation_opportunity,7,8,6,0.75,0.86,0.80
1,risk_safety,7,10,6,0.60,0.86,0.71
2,regulation_governance,14,14,11,0.79,0.79,0.79
3,rights_privacy,4,3,2,0.67,0.50,0.57
4,economy_labour,11,13,9,0.69,0.82,0.75
5,misinformation_integrity,4,1,0,0.00,0.00,NaN
6,none,3,1,1,1.00,0.33,0.50


**To score real labels:** save the labelled sheet (for example `data/validation_sheet_labelled_A.csv`), then run `score_sheet("data/validation_sheet_labelled_A.csv")`. It stops with an error on empty or misspelled labels, and writes `validation_per_frame.csv`, `validation_confusion.csv` and `validation_summary.csv` to `dataset/analysis/`. Two labellers can be compared with `agreement_report(labels_A, labels_B, LABELS)`.

## 3. Dictionary vs the dataset's own frame labels
**These are two different instruments, so disagreement is expected.** The dataset assigns frames from GDELT's *entity-name and quotation fields* (not the article body), keyword match plus an embedding confirmation: a frame is *confirmed* when `kw_* == 1` and `emb_* > 0`; `dominant_frame` is the confirmed frame with the highest normalised keyword count, null if none is confirmed. Our dictionary reads the article body in full (the main scoring, `frame_scores_news_full.csv`, lexicon v1.2; this section used the 300-word cut before 2026-10-29). Agreement here says how far the two instruments move together; it does not make either one the truth.

Compared for the 420 corpus articles, matched on the normalised URL: (a) per frame, our "frame present" (at least 1 hit) against the dataset's "frame confirmed"; (b) our top frame against the dataset's `dominant_frame` where the dataset has one.

In [8]:
import pyarrow.parquet as pq
ds_cols = [c for c in pq.ParquetFile(CFG["parquet"]).schema_arrow.names if c.startswith(("kw_", "emb_"))]
ds = pd.read_parquet(CFG["parquet"], columns=["document_id", "dominant_frame"] + ds_cols)
ds["url"] = ds["document_id"].str.split("?").str[0].str.rstrip("/"); ds = ds.drop_duplicates("url")
MAP = {"economic_competition_labour": "economy_labour"}
ds["dominant_frame"] = ds["dominant_frame"].replace(MAP)
for f in FRAMES:
    kw = "kw_economic_competition_labour" if f == "economy_labour" else f"kw_{f}"; em = "emb_economic_competition_labour" if f == "economy_labour" else f"emb_{f}"
    ds[f"confirmed_{f}"] = ((ds[kw] == 1) & (ds[em] > 0)).astype(int)
cmp_ = news[["doc_id", "url", "title", "region"]].merge(ds[["url", "dominant_frame"] + [f"confirmed_{f}" for f in FRAMES]], on="url", how="left")
sc_nf = pd.read_csv(OUT / "frame_scores_news_full.csv"); assert (sc_nf["doc_id"] == news["doc_id"]).all()
cmp_[HC] = sc_nf[HC].to_numpy(); cmp_["hits_total"] = sc_nf["hits_total"].to_numpy()
cmp_["our_top_frame"] = cmp_.apply(top_frame, axis=1)
print(len(cmp_), "articles;", int(cmp_["dominant_frame"].notna().sum()), "have a dataset dominant_frame;", int(cmp_[f"confirmed_{FRAMES[0]}"].isna().sum()), "not found in the dataset by URL")

420 articles; 304 have a dataset dominant_frame; 0 not found in the dataset by URL


In [9]:
# (a) per frame: ours (>= 1 hit) vs dataset confirmed
rows = []
for f in FRAMES:
    ours = (cmp_[f"hits_{f}"] >= 1).astype(int).to_numpy(); theirs = cmp_[f"confirmed_{f}"].to_numpy()
    tp = int(((ours == 1) & (theirs == 1)).sum())
    rows.append({"frame": f, "dataset_confirmed_%": round(100 * theirs.mean(), 1), "ours_present_%": round(100 * ours.mean(), 1),
                 "agreement_%": round(100 * (ours == theirs).mean(), 1), "kappa": round(cohen_kappa(theirs, ours, [0, 1]), 3),
                 "precision_vs_dataset": round(tp / max(ours.sum(), 1), 3), "recall_vs_dataset": round(tp / max(theirs.sum(), 1), 3)})
per_frame = pd.DataFrame(rows); per_frame.to_csv(OUT / "dataset_label_comparison_per_frame.csv", index=False)
per_frame

,frame,dataset_confirmed_%,ours_present_%,agreement_%,kappa,precision_vs_dataset,recall_vs_dataset
0,innovation_opportunity,8.8,81.2,25.2,0.012,0.094,0.865
1,risk_safety,13.1,87.9,24.3,0.029,0.144,0.964
2,regulation_governance,7.9,93.3,14.0,0.007,0.082,0.970
3,rights_privacy,6.7,61.4,43.8,0.062,0.097,0.893
4,economy_labour,6.9,83.6,22.4,0.017,0.077,0.931
5,misinformation_integrity,2.6,47.9,54.8,0.057,0.055,1.000


In [10]:
# (b) our top frame vs the dataset dominant_frame (articles where the dataset has one)
both = cmp_[cmp_["dominant_frame"].notna()]
rep_d = agreement_report(both["dominant_frame"], both["our_top_frame"], LABELS)
print("articles compared:", rep_d["n"], "| exact agreement:", round(rep_d["accuracy"], 3), "| kappa:", round(rep_d["kappa"], 3))
rep_d["confusion"].to_csv(OUT / "dataset_label_comparison_confusion.csv"); rep_d["per_class"].round(3).to_csv(OUT / "dataset_label_comparison_dominant_frame.csv", index=False)
display(rep_d["confusion"])
rep_d["per_class"].round(3)

articles compared: 304 | exact agreement: 0.421 | kappa: 0.271


checked,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economy_labour,misinformation_integrity,none
reference,,,,,,,
innovation_opportunity,11,14,12,0,12,2,0
risk_safety,1,42,10,5,1,2,0
regulation_governance,3,37,31,7,22,6,0
rights_privacy,2,10,7,15,0,1,0
economy_labour,1,8,7,0,28,0,0
misinformation_integrity,0,3,2,0,1,1,0
none,0,0,0,0,0,0,0


,label,n_reference,n_checked,true_positive,precision,recall,f1
0,innovation_opportunity,51,18,11,0.611,0.216,0.319
1,risk_safety,61,114,42,0.368,0.689,0.480
2,regulation_governance,106,69,31,0.449,0.292,0.354
3,rights_privacy,35,27,15,0.556,0.429,0.484
4,economy_labour,44,64,28,0.438,0.636,0.519
5,misinformation_integrity,7,12,1,0.083,0.143,0.105
6,none,0,0,0,NaN,NaN,NaN


In [11]:
# examples where they disagree (ids, titles and counts only; no article text)
dis = both[both["dominant_frame"] != both["our_top_frame"]].copy()
dis["our_hits_in_dataset_frame"] = [r[f"hits_{r['dominant_frame']}"] for _, r in dis.iterrows()]
ex = dis.sample(min(25, len(dis)), random_state=SEED)[["doc_id", "region", "dominant_frame", "our_top_frame", "hits_total", "our_hits_in_dataset_frame", "title", "url"]]
ex["title"] = ex["title"].str[:80]
ex.to_csv(OUT / "dataset_label_disagreement_examples.csv", index=False)
print(len(dis), "of", len(both), "articles disagree on the top frame; 25 examples saved to dataset/analysis/dataset_label_disagreement_examples.csv")
ex.head(10)

176 of 304 articles disagree on the top frame; 25 examples saved to dataset/analysis/dataset_label_disagreement_examples.csv


,doc_id,region,dominant_frame,our_top_frame,hits_total,our_hits_in_dataset_frame,title,url
34,news_0035,US,economy_labour,risk_safety,25,8,Trump shelves AI oversight order,http://www.heraldglobe.com/news/279069771/trum...
90,news_0091,Other,innovation_opportunity,risk_safety,29,2,Former xAI engineer claims he was fired for ra...,https://www.livemint.com/technology/tech-news/...
336,news_0337,UK,risk_safety,innovation_opportunity,42,9,The AI data paradox: Maximising reward while m...,https://www.bizcommunity.com/article/the-ai-da...
57,news_0058,Other,innovation_opportunity,economy_labour,81,9,Eightco Holdings (NASDAQ: ORBS) Reports Total ...,https://www.manilatimes.net/2026/05/13/tmt-new...
166,news_0167,US,regulation_governance,risk_safety,14,0,Trump's new AI policies keep culture war focus...,https://www.wnmufm.org/2025-07-23/trumps-new-a...
29,news_0030,Other,economy_labour,risk_safety,43,3,Pentagon's Use of Grok Raises AI Security Conc...,https://www.govinfosecurity.com/pentagons-use-...
279,news_0280,US,regulation_governance,economy_labour,20,1,Foundations want to curb AI developers’ influe...,https://www.bostonherald.com/2025/10/14/philan...
403,news_0404,EU,risk_safety,regulation_governance,19,3,The AI-Empowered Professional: Why the Same To...,https://www.pressenza.com/2026/06/the-ai-empow...
256,news_0257,US,regulation_governance,risk_safety,12,2,US judge suspends govt sanctions on AI company...,https://www.digitaljournal.com/business/us-jud...
337,news_0338,UK,regulation_governance,economy_labour,49,9,The rise of ‘unmeritorious’ workplace claims i...,https://www.lawyersweekly.com.au/biglaw/43439-...


### Dictionary vs the dataset labels on the validation windows, next to the full text
The 25 news articles of the validation sheet, scored two ways: on the **window** shown to the labellers, and on the **full text** (the main scoring). Only the articles where the dataset has a dominant frame can be compared, so the numbers are small; the first row repeats the all-article full-text result from above. Saved as `dataset_label_comparison_windows.csv` (one row per article, no text, no label ids) and `dataset_label_comparison_windows_summary.csv`.

In [12]:
kn = key[key["source"] == "news"].set_index("ref")
w = cmp_.set_index("doc_id").loc[kn.index].copy(); w.index.name = "doc_id"
w["our_top_frame_window"] = kn["lexicon_top_frame"]; w["hits_window"] = kn["lexicon_hits"].astype(int)
w["our_top_frame_full_text"] = w["our_top_frame"]; w["hits_full_text"] = w["hits_total"].astype(int)
w = w.join(win_info.set_index("doc_id")[["words", "passes_rule"]].rename(columns={"words": "window_words", "passes_rule": "window_passes_rule"}))
has = w["dominant_frame"].notna()
w["agree_window"] = np.where(has, w["our_top_frame_window"] == w["dominant_frame"], np.nan); w["agree_full_text"] = np.where(has, w["our_top_frame_full_text"] == w["dominant_frame"], np.nan)
w.reset_index().rename(columns={"dominant_frame": "dataset_dominant_frame"})[["doc_id", "region", "dataset_dominant_frame", "our_top_frame_window", "our_top_frame_full_text", "hits_window", "hits_full_text",
    "window_words", "window_passes_rule", "agree_window", "agree_full_text"]].to_csv(OUT / "dataset_label_comparison_windows.csv", index=False)

def scope(name, a, b):
    a, b = list(a), list(b); return {"scope": name, "n_articles": len(a), "agreement": round(float(np.mean([x == y for x, y in zip(a, b)])), 3), "kappa": round(float(cohen_kappa(a, b, LABELS)), 3) if len(a) > 1 else np.nan}
lab = w[has]
summ = pd.DataFrame([scope("full text, all articles with a dataset label", both["dominant_frame"], both["our_top_frame"]),
                     scope("full text, the 25 sheet articles that have a dataset label", lab["dominant_frame"], lab["our_top_frame_full_text"]),
                     scope("window shown to labellers, the same articles", lab["dominant_frame"], lab["our_top_frame_window"])])
same_top = float((w["our_top_frame_window"] == w["our_top_frame_full_text"]).mean())
summ = pd.concat([summ, pd.DataFrame([{"scope": "all 25 sheet articles: window top frame equals full-text top frame (share)", "n_articles": len(w), "agreement": round(same_top, 3), "kappa": np.nan}])], ignore_index=True)
summ.to_csv(OUT / "dataset_label_comparison_windows_summary.csv", index=False)
display(summ)

,scope,n_articles,agreement,kappa
0,"full text, all articles with a dataset label",304,0.421,0.271
1,"full text, the 25 sheet articles that have a d...",19,0.579,0.433
2,"window shown to labellers, the same articles",19,0.368,0.146
3,all 25 sheet articles: window top frame equals...,25,0.680,NaN


## Limits to state in the report
- The dataset's labels come from GDELT entity names and quotations, ours from article text, so the comparison is between two noisy instruments; see the confusion matrix for where they differ.
- The dataset labels only 41% of articles (null = unconfirmed), and its governance base rate is inflated by the corpus's own governance filter (dataset card), so agreement on `regulation_governance` is partly built in.
- Human validation (section 1) is the real test of the dictionary; this comparison is a cross-check, not a replacement.
- The validation sheet shows excerpts, not full texts: the first 300 words of each chatbot answer and a topic-selected 250 to 300 word window of each news article (two of the 25 windows do not pass the on-topic rule because their article has only two regulation words, far apart). The main analysis scores full texts; the full-text scores are checked against the dataset labels (section 3). Nobody has labelled the sheet yet, so there is no human-agreement figure.